In [2]:
# cell 1 — imports and path setup
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path

# paths
RAW_DATA = Path("../data/raw/steam.csv")
DB_PATH = Path("../data/database/steam.db")

print("Libraries loaded")
print(f"Raw data path exists: {RAW_DATA.exists()}")

Libraries loaded
Raw data path exists: True


In [3]:
# cell 2 — first look at raw data
df = pd.read_csv(RAW_DATA)

print(f"Shape: {df.shape}")
print(f"\nColumns:\n{df.columns.tolist()}")
print(f"\nFirst 3 rows:")
df.head(3)

Shape: (27075, 18)

Columns:
['appid', 'name', 'release_date', 'english', 'developer', 'publisher', 'platforms', 'required_age', 'categories', 'genres', 'steamspy_tags', 'achievements', 'positive_ratings', 'negative_ratings', 'average_playtime', 'median_playtime', 'owners', 'price']

First 3 rows:


,appid,name,release_date,english,developer,publisher,platforms,required_age,categories,genres,steamspy_tags,achievements,positive_ratings,negative_ratings,average_playtime,median_playtime,owners,price
0,10,Counter-Strike,2000-11-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,124534,3339,17612,317,10000000-20000000,7.19
1,20,Team Fortress Classic,1999-04-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,3318,633,277,62,5000000-10000000,3.99
2,30,Day of Defeat,2003-05-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Valve Anti-Cheat enabled,Action,FPS;World War II;Multiplayer,0,3416,398,187,34,5000000-10000000,3.99


In [4]:
# cell 3 — data audit
print("=== DTYPES ===")
print(df.dtypes)

print("\n=== NULLS ===")
print(df.isnull().sum())

print("\n=== SAMPLE VALUES (multi-value columns) ===")
print("platforms:    ", df['platforms'].iloc[0])
print("genres:       ", df['genres'].iloc[0])
print("steamspy_tags:", df['steamspy_tags'].iloc[0])
print("categories:   ", df['categories'].iloc[0])
print("owners:       ", df['owners'].iloc[0])

=== DTYPES ===
appid                 int64
name                    str
release_date            str
english               int64
developer               str
publisher               str
platforms               str
required_age          int64
categories              str
genres                  str
steamspy_tags           str
achievements          int64
positive_ratings      int64
negative_ratings      int64
average_playtime      int64
median_playtime       int64
owners                  str
price               float64
dtype: object

=== NULLS ===
appid                0
name                 0
release_date         0
english              0
developer            1
publisher           14
platforms            0
required_age         0
categories           0
genres               0
steamspy_tags        0
achievements         0
positive_ratings     0
negative_ratings     0
average_playtime     0
median_playtime      0
owners               0
price                0
dtype: int64

=== SAMPLE VALUES (multi

In [5]:
# cell 4 — deeper audit
print("=== OWNERS unique values (sample) ===")
print(df['owners'].value_counts().head(10))

print("\n=== RELEASE DATE sample ===")
print(df['release_date'].head(10))

print("\n=== PRICE distribution ===")
print(df['price'].describe())

print("\n=== DUPLICATE appids ===")
print(f"Duplicate appids: {df['appid'].duplicated().sum()}")

=== OWNERS unique values (sample) ===
owners
0-20000              18596
20000-50000           3059
50000-100000          1695
100000-200000         1386
200000-500000         1272
500000-1000000         513
1000000-2000000        288
2000000-5000000        193
5000000-10000000        46
10000000-20000000       21
Name: count, dtype: int64

=== RELEASE DATE sample ===
0    2000-11-01
1    1999-04-01
2    2003-05-01
3    2001-06-01
4    1999-11-01
5    2000-11-01
6    1998-11-08
7    2004-03-01
8    2001-06-01
9    2004-11-16
Name: release_date, dtype: str

=== PRICE distribution ===
count    27075.000000
mean         6.078193
std          7.874922
min          0.000000
25%          1.690000
50%          3.990000
75%          7.190000
max        421.990000
Name: price, dtype: float64

=== DUPLICATE appids ===
Duplicate appids: 0


In [6]:
# cell 5 — structural cleaning

# 1. fix nulls
df['developer'] = df['developer'].fillna('Unknown')
df['publisher'] = df['publisher'].fillna('Unknown')

# 2. convert price from GBP to USD
GBP_TO_USD = 1.27
df['price'] = (df['price'] * GBP_TO_USD).round(2)

# 3. parse release_date
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_year'] = df['release_date'].dt.year
df['release_month'] = df['release_date'].dt.month

# 4. engineer owner_midpoint from range string
def parse_owners(owner_str):
    low, high = owner_str.split('-')
    return (int(low) + int(high)) // 2

df['owner_midpoint'] = df['owners'].apply(parse_owners)

# 5. engineer total_ratings and positive_ratio
df['total_ratings'] = df['positive_ratings'] + df['negative_ratings']
df['positive_ratio'] = np.where(
    df['total_ratings'] > 0,
    df['positive_ratings'] / df['total_ratings'],
    np.nan
)

# 6. engineer success_score (reach + engagement only)
df['success_score'] = (
    np.log1p(df['owner_midpoint']) * 0.7 +
    np.log1p(df['average_playtime']) * 0.3
)

print("=== CLEANING DONE ===")
print(df[['name', 'owner_midpoint', 'price', 'positive_ratio', 'success_score']].head(5))
print(f"\nNew shape: {df.shape}")

=== CLEANING DONE ===
                        name  owner_midpoint  price  positive_ratio  \
0             Counter-Strike        15000000   9.13        0.973888   
1      Team Fortress Classic         7500000   5.07        0.839787   
2              Day of Defeat         7500000   5.07        0.895648   
3         Deathmatch Classic         7500000   5.07        0.826623   
4  Half-Life: Opposing Force         7500000   5.07        0.947996   

   success_score  
0      14.499410  
1      12.769576  
2      12.652222  
3      12.748338  
4      13.012615  

New shape: (27075, 24)


In [7]:
# cell 6 — write to SQLite
import os
os.makedirs(DB_PATH.parent, exist_ok=True)  # creates data/database/ if it doesn't exist

conn = sqlite3.connect(DB_PATH)

# table 1: games (one row per game, all core columns)
games_cols = [
    'appid', 'name', 'release_date', 'release_year', 'release_month',
    'developer', 'publisher', 'required_age', 'achievements',
    'positive_ratings', 'negative_ratings', 'total_ratings', 'positive_ratio',
    'average_playtime', 'median_playtime', 'owner_midpoint', 'price',
    'success_score', 'english'
]

df[games_cols].to_sql('games', conn, if_exists='replace', index=False)
print(f"games table: {len(df)} rows written")

# table 2: game_platforms (one row per platform per game)
platforms_rows = []
for _, row in df.iterrows():
    for platform in str(row['platforms']).split(';'):
        platforms_rows.append({'appid': row['appid'], 'platform': platform.strip()})

pd.DataFrame(platforms_rows).to_sql('game_platforms', conn, if_exists='replace', index=False)
print(f"game_platforms table: {len(platforms_rows)} rows written")

# table 3: game_genres (one row per genre per game)
genre_rows = []
for _, row in df.iterrows():
    for genre in str(row['genres']).split(';'):
        genre_rows.append({'appid': row['appid'], 'genre': genre.strip()})

pd.DataFrame(genre_rows).to_sql('game_genres', conn, if_exists='replace', index=False)
print(f"game_genres table: {len(genre_rows)} rows written")

# table 4: game_tags (one row per tag per game)
tag_rows = []
for _, row in df.iterrows():
    for tag in str(row['steamspy_tags']).split(';'):
        tag_rows.append({'appid': row['appid'], 'tag': tag.strip()})

pd.DataFrame(tag_rows).to_sql('game_tags', conn, if_exists='replace', index=False)
print(f"game_tags table: {len(tag_rows)} rows written")

conn.close()
print("\nDatabase written to:", DB_PATH)

games table: 27075 rows written
game_platforms table: 40371 rows written
game_genres table: 76462 rows written
game_tags table: 78002 rows written

Database written to: ..\data\database\steam.db


In [8]:
# cell 7 — verify database
conn = sqlite3.connect(DB_PATH)

tables = ['games', 'game_platforms', 'game_genres', 'game_tags']
for table in tables:
    count = pd.read_sql(f"SELECT COUNT(*) as rows FROM {table}", conn).iloc[0,0]
    print(f"{table}: {count} rows")

print("\n=== SAMPLE FROM games ===")
pd.read_sql("SELECT appid, name, price, owner_midpoint, positive_ratio, success_score FROM games LIMIT 5", conn)

games: 27075 rows
game_platforms: 40371 rows
game_genres: 76462 rows
game_tags: 78002 rows

=== SAMPLE FROM games ===


,appid,name,price,owner_midpoint,positive_ratio,success_score
0,10,Counter-Strike,9.13,15000000,0.973888,14.499410
1,20,Team Fortress Classic,5.07,7500000,0.839787,12.769576
2,30,Day of Defeat,5.07,7500000,0.895648,12.652222
3,40,Deathmatch Classic,5.07,7500000,0.826623,12.748338
4,50,Half-Life: Opposing Force,5.07,7500000,0.947996,13.012615
